# REINFORCE（基础策略梯度算法）

> Williams, 1992. 策略梯度族的开山算法，PPO/GRPO/A2C 均由其派生。

## 1. 题目背景与动机
在 RLHF 出现之前，让"参数化策略 $\pi_\theta$ 直接最大化期望回报"的最朴素做法就是 **REINFORCE**。
它直接套用**策略梯度定理**：
$$\nabla_\theta J(\theta)=\mathbb E_{\tau\sim\pi_\theta}\Big[\sum_t \nabla_\theta\log\pi_\theta(a_t|s_t)\,G_t\Big]$$
其中 $G_t=\sum_{k\ge t}\gamma^{k-t}r_k$ 是从 $t$ 时刻起的回报（return）。

在 LLM 对齐语境下：
- $s_t=(x, y_{<t})$：prompt + 已生成 token；
- $a_t=y_t$：下一步生成 token；
- $r$：通常只在序列末尾由 RM/规则给一个标量奖励，中间步为 0；
- $G_t$ 简化为 "序列总奖励"（无折扣或仅序列级）。

## 2. 损失
最大化 $J$ 等价于最小化：
$$\mathcal L_{REINFORCE}=-\mathbb E\big[G\cdot\log\pi_\theta(y|x)\big]$$
为降方差常引入 baseline $b$（如滑动平均、批均值）：
$$\mathcal L=-\mathbb E\big[(G-b)\cdot\log\pi_\theta(y|x)\big]$$

## 3. 与后续算法关系
- 加 critic → A2C / A3C；
- 加 clip + 多 epoch 复用 → PPO；
- 把 critic 换成组内均值 → GRPO；
- 把 RL 目标改写成偏好对闭式解 → DPO。

## 4. 考察点
- 策略梯度定理推导（log-derivative trick）
- on-policy 含义、为何不能复用旧样本
- baseline 不引入偏差、只降方差
- 高方差问题与熵正则


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from transformers import LlamaConfig, LlamaForCausalLM

torch.manual_seed(42)

# ============================================================
# REINFORCE 核心思想：
# 直接用 策略梯度定理 ∇J = E[ ∇log π(a|s) * G ]
# 没有 critic、没有 clip、没有重要性采样（严格 on-policy）
# 是 PPO / GRPO / A2C 的鼻祖
# ============================================================

# 超参数
BATCH = 4          # 4 条独立轨迹（4 个 prompt 各生成 1 个回答）
VOCAB_SIZE = 12
PROMPT_LEN = 6
OUTPUT_LEN = 4

# 构造数据：BATCH 个不同 prompt，各生成 1 个回答（REINFORCE 不需要组采样）
prompts = torch.randint(0, VOCAB_SIZE, (BATCH, PROMPT_LEN))
outputs = torch.randint(0, VOCAB_SIZE, (BATCH, OUTPUT_LEN))
full_ids = torch.cat([prompts, outputs], dim=1)         # [B, 10]
full_mask = torch.ones_like(full_ids)                   # [B, 10]

# 响应掩码：仅在生成部分计算损失
response_mask = torch.zeros_like(full_ids)
response_mask[:, PROMPT_LEN:] = 1                       # [B, 10]

print("== REINFORCE 数据形状 ==")
print("批次大小:", BATCH, "（注意：REINFORCE 不需要组采样，每 prompt 1 条轨迹）")
print("完整序列:", full_ids.shape, "  响应掩码:", response_mask.shape)


In [ ]:
# 策略模型（仅此一个模型，无 critic、无 reference）
policy_model = LlamaForCausalLM(config=LlamaConfig(
    vocab_size=VOCAB_SIZE, num_hidden_layers=1, hidden_size=32
))

print("== 模型清单 ==")
print("REINFORCE 只需 1 个模型: Policy  ✓")
print("PPO     需要 4 个模型: Actor + Critic + Reward + Reference")
print("GRPO    需要 2 个模型: Policy + Reference")
print("DPO     需要 2 个模型: Policy + Reference（offline）")


In [ ]:
# ============================================================
# 工具函数：token 级对数概率
# ============================================================
def logprobs_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """从 logits 取出 labels 对应位置的 log p
    logits: [B, T, V], labels: [B, T] -> [B, T]
    """
    logp = F.log_softmax(logits, dim=-1)
    logp_labels = torch.gather(logp, dim=-1, index=labels.unsqueeze(-1))
    return logp_labels.squeeze(-1)

def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return (values * mask).sum() / mask.sum().clamp(min=1e-8)

# 前向计算策略对数概率
policy_logits = policy_model(full_ids).logits            # [B, T, V]
policy_logprobs = logprobs_from_logits(policy_logits, full_ids)  # [B, T]

print("策略 logits 形状:", policy_logits.shape)
print("策略 logp  形状:", policy_logprobs.shape)


In [ ]:
# ============================================================
# 奖励函数：序列级标量奖励
# REINFORCE 经典设定：只在轨迹末尾给一个奖励 G
# 这里用一个小 MLP 模拟 RM，对每个序列输出一个标量
# ============================================================
class RewardModel(nn.Module):
    def __init__(self, vocab_size: int = 12, hidden_size: int = 8) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(input_ids)
        outputs, _ = self.lstm(x)
        last_hidden = outputs[:, -1]                  # [B, hidden]
        return self.head(last_hidden).squeeze(-1)     # [B]

reward_model = RewardModel()
with torch.no_grad():
    rewards = reward_model(full_ids)                   # [B]

print("== 序列级奖励 ==")
print("奖励形状:", rewards.shape, "  # [B]，每条轨迹一个标量")
print("奖励值:", rewards.detach().tolist())


In [ ]:
# ============================================================
# 回报计算 G_t
# 由于奖励只在序列末尾给出，对生成部分所有 token 而言 G_t = R（同一值）
# 若有中间奖励，则用 G_t = r_t + γ * G_{t+1} 反向累计
# ============================================================
def compute_returns(rewards: torch.Tensor, response_mask: torch.Tensor, gamma: float = 1.0) -> torch.Tensor:
    """
    将序列级标量奖励广播为 token 级回报
    rewards: [B] -> returns: [B, T]
    """
    returns = rewards.unsqueeze(1).expand_as(response_mask).float()
    returns = returns * response_mask
    return returns

returns = compute_returns(rewards, response_mask)
print("== token 级回报 G_t ==")
print("形状:", returns.shape)
print("第 0 条轨迹的 token 回报:", returns[0].tolist())
print("（生成部分共享同一序列级奖励，prompt 部分为 0）")


In [ ]:
# ============================================================
# Baseline：滑动平均
# 关键性质：baseline 不依赖动作时，E[∇log π * b] = 0，不引入偏差
# 只降方差。常用实现：running mean of returns
# ============================================================
class MovingAverageBaseline:
    def __init__(self, init_value: float = 0.0, momentum: float = 0.95) -> None:
        self.value = init_value
        self.momentum = momentum

    def update(self, batch_returns: torch.Tensor) -> None:
        # batch_returns: [B] 标量奖励
        batch_mean = batch_returns.detach().mean().item()
        self.value = self.momentum * self.value + (1 - self.momentum) * batch_mean
        return self.value

baseline = MovingAverageBaseline(init_value=0.0)
b = baseline.update(rewards)
print("== Baseline ==")
print(f"滑动平均 baseline b = {b:.6f}")
print(f"减去 baseline 后的有效回报: {(rewards - b).detach().tolist()}")


In [ ]:
# ============================================================
# REINFORCE 损失
#
# L = -E[ (G - b) * log π_θ(y|x) ] - β * H(π)
#
#   - 第一项：策略梯度（最大化期望回报）
#   - 第二项：熵正则 H(π) = -Σ π log π，鼓励探索，防止过早坍缩
#
# 注意：严格 on-policy，不使用重要性采样比率
# ============================================================
def reinforce_loss(policy_logits: torch.Tensor, policy_logprobs: torch.Tensor, returns: torch.Tensor, response_mask: torch.Tensor, baseline_value: float = 0.0, entropy_coef: float = 0.01) -> torch.Tensor:
    """
    policy_logits:   [B, T, V] 当前策略 logits（用于熵）
    policy_logprobs: [B, T]    当前策略对数概率（需要梯度）
    returns:         [B, T]    token 级回报
    response_mask:   [B, T]    生成部分掩码
    baseline_value:  float     标量 baseline
    entropy_coef:    float     熵正则系数 β
    """
    # 1. 优势 = 回报 - baseline
    advantages = (returns - baseline_value) * response_mask   # [B, T]

    # 2. 策略梯度损失：-A * log π
    pg_loss_token = -advantages * policy_logprobs             # [B, T]
    pg_loss = masked_mean(pg_loss_token, response_mask)

    # 3. 熵正则（在生成部分）
    probs = F.softmax(policy_logits, dim=-1)
    entropy_token = -torch.sum(probs * F.log_softmax(policy_logits, dim=-1), dim=-1)
    entropy = masked_mean(entropy_token, response_mask)

    # 4. 总损失 = 策略损失 - 熵奖励
    total_loss = pg_loss - entropy_coef * entropy

    stats = {
        "pg_loss": pg_loss.item(),
        "entropy": entropy.item(),
        "total_loss": total_loss.item(),
        "baseline": baseline_value,
    }
    return total_loss, stats

# 测试损失计算
total_loss, stats = reinforce_loss(
    policy_logits, policy_logprobs, returns, response_mask,
    baseline_value=baseline.value, entropy_coef=0.01
)

print("== REINFORCE 损失 ==")
for k, v in stats.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# 单步训练流程演示
# REINFORCE 严格 on-policy：必须用当前策略采样的数据，不能复用旧样本
# ============================================================
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4)

print("== REINFORCE 单步训练 ==")
print()

# Step 1: 采样轨迹（无梯度）
print("[Step 1] 用当前策略采样轨迹（此处用预构造数据演示）")
with torch.no_grad():
    sampled_rewards = reward_model(full_ids)
print(f"  采样奖励: {sampled_rewards.tolist()}")

# Step 2: 更新 baseline
print()
print("[Step 2] 更新滑动平均 baseline")
b = baseline.update(sampled_rewards)
print(f"  baseline = {b:.6f}")

# Step 3: 计算回报
print()
print("[Step 3] 计算 token 级回报")
tok_returns = compute_returns(sampled_rewards, response_mask)

# Step 4: 前向（需要梯度）
print()
print("[Step 4] 策略前向传播（梯度开启）")
new_logits = policy_model(full_ids).logits
new_logprobs = logprobs_from_logits(new_logits, full_ids)

# Step 5: 计算 REINFORCE 损失
print()
print("[Step 5] 计算 REINFORCE 损失")
loss, train_stats = reinforce_loss(
    new_logits, new_logprobs, tok_returns, response_mask,
    baseline_value=baseline.value, entropy_coef=0.01
)

# Step 6: 反向传播 + 更新
print()
print("[Step 6] 反向传播 + 参数更新")
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
optimizer.step()
print(f"  梯度范数: {grad_norm.item():.6f}")
print()
print("== 训练统计 ==")
for k, v in train_stats.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# REINFORCE vs PPO vs GRPO 对比
# ============================================================
summary = """
╔══════════════════════╦═══════════════════════════╦════════════════════════════╗
║        维度          ║        REINFORCE          ║        PPO / GRPO           ║
╠══════════════════════╬═══════════════════════════╬════════════════════════════╣
║ 模型数量             ║ 1 (Policy)                ║ 2~4 (含 Critic/Ref/RM)     ║
║ 优势估计             ║ G - b (轨迹回报-基线)     ║ GAE / 组内相对              ║
║ 重要性采样           ║ 无（严格 on-policy）      ║ 有 ratio = π/π_old         ║
║ 数据复用             ║ 1 epoch 即丢弃            ║ 多 epoch 复用（PPO）        ║
║ Clip                 ║ 无                        ║ 有（防策略崩溃）            ║
║ 方差                 ║ 高（无 critic）           ║ 低（critic / 组内归一）     ║
║ 适用场景             ║ 教学/简单任务             ║ 大规模 RLHF                 ║
║ 代表工作             ║ Williams 1992             ║ InstructGPT / DeepSeek-R1  ║
╚══════════════════════╩═══════════════════════════╩════════════════════════════╝

REINFORCE 关键公式:
  策略梯度: ∇J = E[ ∇log π(a|s) * G ]
  损失:     L = -E[ (G - b) * log π(y|x) ] - β * H(π)

演进路线:
  REINFORCE → A2C(加 critic) → PPO(加 clip+多epoch) → GRPO(组内相对,省 critic)
                                                          ↘ GSPO(序列级重要性)
"""
print(summary)


## ✅ 测试验证

In [ ]:
# 验证 REINFORCE 核心性质
import torch

# 1. 损失为有限值
assert not torch.isnan(torch.tensor(stats["total_loss"])), "loss is NaN"
assert not torch.isinf(torch.tensor(stats["total_loss"])), "loss is Inf"

# 2. 策略梯度方向正确: 奖励为正时，应增大对应 token 的 log prob
# 模拟: 单个 token，reward > 0，梯度应使 log prob 增大
logits = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
logp = torch.log_softmax(logits, dim=-1)
reward = 1.0  # 正奖励
loss = -reward * logp[2]  # 最大化 reward * log p(token=2)
loss.backward()
# 梯度方向: token 2 的 logit 应增大（梯度为负 → 下降 → logit 增大）
assert logits.grad[2] < 0, f"positive reward should increase logit, grad={logits.grad}"

# 3. baseline 不引入偏差: E[∇log π * b] = 0
# 简化验证: 对所有动作求和，baseline 项梯度之和为 0
logits2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
logp2 = torch.log_softmax(logits2, dim=-1)
b = 5.0  # 任意 baseline
baseline_loss = -b * logp2.sum()  # sum over all actions
baseline_loss.backward()
# softmax 的梯度对所有 logit 求和为 0
assert abs(logits2.grad.sum().item()) < 1e-6, f"baseline gradient sum should be 0: {logits2.grad.sum()}"

print("✅ REINFORCE 测试通过: 损失有限、正奖励增大 logit、baseline 零偏差")
